In [ ]:
# imports
import sys
import numpy as np
import pandas as pd
import scipy.linalg as slin

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
plt.style.use('scifigs.mplstyle')

sys.path.append('helpers/pcca_fa/')
import helpers.pcca_fa.pcca_fa_mdl as pf
from dual_pfc_funcs import getParams, load_dict, nansem

# params
SAVE_FIG = False
data_path = 'preprocessed_data/'
params = getParams()
subjects, symbols, color_map = params['subjects'], params['markers'], params['color_map']

In [ ]:
pccafa = {}
for sub in subjects:
    pccafa = {**pccafa, **load_dict(data_path + sub + '_pccafa_cv15dim.pkl')}
fnames = pccafa.keys()
nsess = len(fnames)

In [ ]:
# "eigenspectra"
max_dim = 15
prop_var_across_x1 = np.full(shape=(nsess,max_dim),fill_value=np.nan)
prop_var_across_x2 = np.full(shape=(nsess,max_dim),fill_value=np.nan)

for i, (sess, curr_dat) in enumerate(pccafa.items()):
    p = curr_dat['params'].copy()
    mdl = pf.pcca_fa()
    mdl.set_params(p)
    (canon_dir_x1,canon_dir_x2),_ = mdl.get_canonical_directions()
    curr_d = p['d']

    # normalize the canonical directions to be unit vectors
    norm_canon_dir_x1 = canon_dir_x1 / slin.norm(canon_dir_x1,axis=0)
    norm_canon_dir_x2 = canon_dir_x2 / slin.norm(canon_dir_x2,axis=0)

    for j in range(curr_d):
        prop_var_across_x1[i,j] = norm_canon_dir_x1[:,j].T @ (p['W_1'] @ p['W_1'].T) @ norm_canon_dir_x1[:,j]
        prop_var_across_x2[i,j] = norm_canon_dir_x2[:,j].T @ (p['W_2'] @ p['W_2'].T) @ norm_canon_dir_x2[:,j]

    # turn into proportion
    prop_var_across_x1[i,:] = prop_var_across_x1[i,:] / np.nansum(prop_var_across_x1[i,:])
    prop_var_across_x2[i,:] = prop_var_across_x2[i,:] / np.nansum(prop_var_across_x2[i,:])

fig,ax = plt.subplots(1,3,sharex=True,sharey=True,figsize=(6.5,2))
fig.tight_layout(pad=2.5)
fig.supylabel('$\%$ of across-area sv',color=color_map['across'])
fig.supxlabel('canonical direction (ordered by correlation)')

ms = 3
for j,(sub,sym) in enumerate(zip(subjects,symbols)):
    sub_prefix = sub[:2].title()
    filt = [sub_prefix in s for s in fnames] # same order as array

    curr_x1 = prop_var_across_x1[filt,:] * 100
    ax[j].errorbar(np.arange(curr_x1.shape[1])+1, np.nanmean(curr_x1,axis=0), yerr=nansem(curr_x1,axis=0), fmt='-', marker=sym, color=color_map['across'], ms=ms, capsize=2)
    
    curr_x2 = prop_var_across_x2[filt,:] * 100
    ax[j].errorbar(np.arange(curr_x2.shape[1])+1, np.nanmean(curr_x2,axis=0), yerr=nansem(curr_x2,axis=0), fmt='--', mfc='none', marker=sym, color=color_map['across'], ms=ms, capsize=2)

    ax[j].set_title('Monkey {}'.format(sub_prefix))

ax[0].set_ylim([-5,100])
ax[0].set_xticks(np.arange(4,max_dim,4))
ax[0].set_yticks([0,50,100])
ax[0].set_xlim((0.0, 15.7))

if SAVE_FIG:
    pdf = PdfPages('figs/eig_across.pdf')
    pdf.savefig(fig)
    pdf.close()
else:
    fig.show()

In [ ]:
# eigenspectra
max_dim = 15
prop_var_within_x1 = np.full(shape=(nsess,max_dim),fill_value=np.nan)
prop_var_within_x2 = np.full(shape=(nsess,max_dim),fill_value=np.nan)

for i, (sess, curr_dat) in enumerate(pccafa.items()):
    p = curr_dat['params'].copy()

    # L_1
    vals = slin.svdvals(p['L_1'] @ p['L_1'].T) # eigenvalues
    total_var = np.trace(p['L_1'] @ p['L_1'].T)
    prop_var_within_x1[i,:p['d1']] = vals[:p['d1']] / total_var

    # L_2
    vals = slin.svdvals(p['L_2'] @ p['L_2'].T) # eigenvalues
    total_var = np.trace(p['L_2'] @ p['L_2'].T)
    prop_var_within_x2[i,:p['d2']] = vals[:p['d2']] / total_var


fig,ax = plt.subplots(1,3,sharex=True,sharey=True,figsize=(6.5,2))
fig.tight_layout(pad=2.5)
fig.supylabel('$\%$ of within-area sv',color=color_map['within'])
fig.supxlabel('within-area co-fluctuation pattern (ordered by shared variance)')

ms = 3
for j,(sub,sym) in enumerate(zip(subjects,symbols)):
    sub_prefix = sub[:2].title()
    filt = [sub_prefix in s for s in fnames] # same order as array

    curr_x1 = prop_var_within_x1[filt,:] * 100
    ax[j].errorbar(np.arange(curr_x1.shape[1])+1, np.nanmean(curr_x1,axis=0), yerr=nansem(curr_x1,axis=0), fmt='-', marker=sym, color=color_map['within2'], ms=ms, capsize=2)
    
    curr_x2 = prop_var_within_x2[filt,:] * 100
    ax[j].errorbar(np.arange(curr_x2.shape[1])+1, np.nanmean(curr_x2,axis=0), yerr=nansem(curr_x2,axis=0), fmt='--', mfc='none', marker=sym, color=color_map['within1'], ms=ms, capsize=2)

    ax[j].set_title('Monkey {}'.format(sub_prefix))

ax[0].set_ylim([-5,100])
ax[0].set_xticks(np.arange(4,max_dim,4))
ax[0].set_yticks([0,50,100])
ax[0].set_xlim((0.0, 15.7))

if SAVE_FIG:
    pdf = PdfPages('figs/eig_within.pdf')
    pdf.savefig(fig)
    pdf.close()
else:
    fig.show()